In [0]:
from pyspark.sql.functions import current_timestamp, col

df = (
    spark.read
        .format("json")
        .option("multiLine", True)
        .load("/Volumes/techdados/bronze/vol_landing/customers.json")
        .withColumn("_source_file", col("_metadata.file_path"))
        .withColumn("_ingestion_timestamp", current_timestamp())
)

display(df)

In [0]:
(
    df.write
      .option("overwriteSchema", "true")
      .mode("overwrite")
      .option("comment", "Tabela Bronze - Customers raw do JSON")
      .saveAsTable("techdados.bronze.customers")
)

In [0]:
(
    df.writeTo("techdados.bronze.customers")
      .option("overwriteSchema", "true")
      .option("comment", "Tabela Bronze - Customers raw do JSON")
      .createOrReplace()
)

In [0]:
%sql
-- Criar tabela Bronze via SQL
CREATE OR REPLACE TABLE techdados.bronze.sql_customers
COMMENT 'Tabela Bronze - Customers raw do JSON'
AS SELECT 
  *,
  _metadata.file_path AS _source_file,
  current_timestamp() AS _ingestion_timestamp
FROM read_files('/Volumes/techdados/bronze/vol_landing/customers.json', format => 'json', multiLine => true)

## Referências
- [Método read_files](https://learn.microsoft.com/pt-br/azure/databricks/sql/language-manual/functions/read_files)
- [Documentação Spark](https://spark.apache.org/docs/4.0.0/api/python/index.html)